In [ ]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine,text,inspect
from  sqlalchemy.engine import URL
from snowflake.sqlalchemy import URL
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from snowflake.connector.pandas_tools import pd_writer


**define and configure source and destination engine**

In [2]:
source_url =sqlalchemy.engine.URL.create(
    "postgresql+psycopg2",   # dialect + driver
    username ='postgres',
    password = 'QztLWcoQoMcFPKoKgCyuDwUPjOHUYdtO',
    host = 'trolley.proxy.rlwy.net',
    port = '50067',
    database = 'railway'
)


In [3]:
source_eng = create_engine(source_url)

In [4]:
destination_eng = create_engine(URL( user = 'ogee',
                            password = 'Ogechiokoro9712',
                            account ='JOSCNEC-AL39383', 
                            warehouse ='my_warehouse',
                            database = 'Bronze',
                            schema ='raw'
                          ))

**testing connections**

In [5]:
try:
    with source_eng.connect() as conn:
        print("postgres connection successful")
    with destination_eng.connect() as conn:
        print("snowflake connection successful")
except Exception as e:
    print("postgres connection not successfull!!!")


postgres connection successful
snowflake connection successful


In [25]:
# to get all tables in the source database
table_names = inspect(source_eng).get_table_names()
table_names

['categories',
 'products',
 'customers',
 'orders',
 'coupons',
 'order_items',
 'payments',
 'shipping',
 'inventory',
 'reviews',
 'categories_copy',
 'categories_okiki']

In [26]:
#filtering out unwanted tables
unwanted_tables = ['categories_copy','categories_okiki']
filtered_table = []
for table in table_names:
    if table not in unwanted_tables:
        filtered_table.append(table)
        

In [36]:
filtered_table

['categories',
 'products',
 'customers',
 'orders',
 'coupons',
 'order_items',
 'payments',
 'shipping',
 'inventory',
 'reviews']

In [ ]:
#df = pd.read_sql(q,name),, customers, select * from customers

In [71]:

query = {}
for t in filtered_table:
    query[t] = f"select * from {t}"
    
print(query)
    
    

{'categories': 'select * from categories', 'products': 'select * from products', 'customers': 'select * from customers', 'orders': 'select * from orders', 'coupons': 'select * from coupons', 'order_items': 'select * from order_items', 'payments': 'select * from payments', 'shipping': 'select * from shipping', 'inventory': 'select * from inventory', 'reviews': 'select * from reviews'}


In [77]:
stg_name = {}
for name,q in query.items():
    new_name = f"stg_{name}"
    stg_name[new_name] =q





    
 


   
    


In [78]:
stg_name

{'stg_categories': 'select * from categories',
 'stg_products': 'select * from products',
 'stg_customers': 'select * from customers',
 'stg_orders': 'select * from orders',
 'stg_coupons': 'select * from coupons',
 'stg_order_items': 'select * from order_items',
 'stg_payments': 'select * from payments',
 'stg_shipping': 'select * from shipping',
 'stg_inventory': 'select * from inventory',
 'stg_reviews': 'select * from reviews'}

In [80]:
query = {f"stg_{name}": q for name,q in query.items()}

In [81]:
query

{'stg_categories': 'select * from categories',
 'stg_products': 'select * from products',
 'stg_customers': 'select * from customers',
 'stg_orders': 'select * from orders',
 'stg_coupons': 'select * from coupons',
 'stg_order_items': 'select * from order_items',
 'stg_payments': 'select * from payments',
 'stg_shipping': 'select * from shipping',
 'stg_inventory': 'select * from inventory',
 'stg_reviews': 'select * from reviews'}

**creating dataframe for each table**

In [84]:


dataframe = {}  # keep all DataFrames here
for table, q in query.items():
    dataframe[table] = pd.read_sql(q,source_eng)





    
    


In [86]:
print(dataframe['stg_categories'])

   id            name          slug  is_active  \
0   1     Electronics   electronics       True   
1   2        Clothing      clothing       True   
2   3           Books         books       True   
3   4  Home & Kitchen  home-kitchen       True   
4   5          Sports        sports       True   

                        created_at                       updated_at  
0 2026-03-11 02:31:45.075615+00:00 2026-03-11 02:31:45.075615+00:00  
1 2026-03-11 02:31:45.075615+00:00 2026-03-11 02:31:45.075615+00:00  
2 2026-03-11 02:31:45.075615+00:00 2026-03-11 02:31:45.075615+00:00  
3 2026-03-11 02:31:45.075615+00:00 2026-03-11 02:31:45.075615+00:00  
4 2026-03-11 02:31:45.075615+00:00 2026-03-11 02:31:45.075615+00:00  


**loading data to snowflake**

In [88]:
for name, df in dataframe.items():
    df.to_sql(name,destination_eng, index = False)
    print(f"successfully moved {name} to snowflake")

successfully moved stg_categories to snowflake
successfully moved stg_products to snowflake
successfully moved stg_customers to snowflake
successfully moved stg_orders to snowflake
successfully moved stg_coupons to snowflake
successfully moved stg_order_items to snowflake
successfully moved stg_payments to snowflake
successfully moved stg_shipping to snowflake
successfully moved stg_inventory to snowflake
successfully moved stg_reviews to snowflake
